# LLAMA 3.2 1B Fine-tuning

In [1]:
import torch
import dataset_downloader
from datasets import load_dataset, DatasetDict
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from trl import SFTConfig, SFTTrainer

In [2]:
MODEL_ID = "meta-llama/Llama-3.2-3B"

In [3]:

def find_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    elif torch.cuda.is_available():
        return "cuda"
    else:
        return "cpu"

def check_cuda() -> bool:
    device = find_device()
    if device == "cuda":
        print(f"✅ CUDA is available. Found {torch.cuda.device_count()} device(s).")
        print(f"Device Name: {torch.cuda.get_device_name(0)}")
        print("Will use quantizaiton")
        return True
    else:
        print(f"{'❌' if device == 'cpu' else '⚠️'} CUDA is NOT available. Skipping quantization, using {device}")
        return False

CUDA_AVAILABLE = check_cuda()
DEVICE = find_device()


⚠️ CUDA is NOT available. Skipping quantization, using mps


In [4]:
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/venky73/spam-mails-dataset"
local_dataset_uri = dataset_downloader.download_and_unzip_dataset(dataset_url, "datasets/spam-dataset-enron1")
column_names = ['id', 'label', 'text', 'class']

Download complete.
Extracting to /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/datasets...
Extraction complete.
Cleaned up: Removed /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/datasets/spam-dataset-enron1.zip
Found largest CSV in zip: /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/datasets/spam_ham_dataset.csv (5.25 MB)


In [5]:
dataset = load_dataset("csv", data_files=local_dataset_uri, split="train")

train_testvalid = dataset.train_test_split(test_size=0.2, seed=67)
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=67)

dataset = DatasetDict({
    "train": train_testvalid["train"],
    "test": test_valid["test"],
    "validation": test_valid["train"]
})

mapping = {}
current_columns = dataset["train"].column_names

for i in range(len(column_names)):
    mapping[current_columns[i]] = column_names[i]

dataset = dataset.rename_columns(mapping)

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
def dataset_transform(sample):
    raw_response = 'valid' if sample['class'] == 0 else 'spam'
    raw_email_text = sample['text']  # Get the original email text

    # For base models, format as plain text instead of chat messages
    # Using a simple prompt format that the model can learn
    instruction = 'Classify the message as either `valid` or `spam` do not add anything else into the response'
    
    # Format: Instruction\n\nEmail: {email}\n\nClassification: {response}
    formatted_text = f"""Instruction: {instruction}

Email: {raw_email_text}

Classification: {raw_response}"""
    
    return {
        'text': formatted_text,
        'label': raw_response
    }

dataset = dataset.map(dataset_transform, remove_columns=column_names)


Map:   0%|          | 0/4136 [00:00<?, ? examples/s]

Map:   0%|          | 0/518 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

In [7]:
def balance_dataset_split(dataset_split):
    spam_samples = [sample for sample in dataset_split if sample['label'] == 'spam']
    valid_samples = [sample for sample in dataset_split if sample['label'] == 'valid']
    
    min_count = min(len(spam_samples), len(valid_samples))
    
    print(f"Original - Spam: {len(spam_samples)}, Valid: {len(valid_samples)}")
    print(f"Balanced - Using {min_count} samples from each class")
    
    balanced_samples = spam_samples[:min_count] + valid_samples[:min_count]
    
    import random
    random.seed(67)
    random.shuffle(balanced_samples)
    
    return balanced_samples

train_balanced = balance_dataset_split(dataset['train'])
test_balanced = balance_dataset_split(dataset['test'])
validation_balanced = balance_dataset_split(dataset['validation'])

from datasets import Dataset
dataset = DatasetDict({
    "train": Dataset.from_list(train_balanced),
    "test": Dataset.from_list(test_balanced),
    "validation": Dataset.from_list(validation_balanced)
})

print(f"\nFinal dataset sizes:")
print(f"Train: {len(dataset['train'])}")
print(f"Test: {len(dataset['test'])}")
print(f"Validation: {len(dataset['validation'])}")

Original - Spam: 1191, Valid: 2945
Balanced - Using 1191 samples from each class
Original - Spam: 162, Valid: 356
Balanced - Using 162 samples from each class
Original - Spam: 146, Valid: 371
Balanced - Using 146 samples from each class

Final dataset sizes:
Train: 2382
Test: 324
Validation: 292


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

if CUDA_AVAILABLE:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        use_cache=False
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": DEVICE},
    dtype=torch.float16,
    use_cache=False
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_BIAS = "none"

In [10]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# For base models (non-instruct), we just need basic tokenizer setup
# Set pad token to EOS token (standard practice for LLAMA models)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Update model config to match tokenizer
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [12]:
OUTPUT_DIR = "{MODEL_ID}_finetune"
MAX_LENGTH = 512
if CUDA_AVAILABLE:
    print("✅ CUDA Detected: Using NVIDIA-optimized settings (8-bit optimizer, Liger kernel).")
    target_optim = "paged_adamw_8bit"
    target_liger_kernel = True  # Only on linux
else:
    print("🍎 CUDA Not Available (likely Mac/MPS): Using MPS-friendly settings (standard optimizer, No Liger).")
    target_optim = "adamw_torch"
    target_liger_kernel = False

# Define Training Arguments
training_args = SFTConfig(
    output_dir="./results",
    logging_steps=10,
    disable_tqdm=False,
    report_to="none",

    # Training schedule / optimization
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=1,
    max_steps=800,
    learning_rate=5e-5,

    optim=target_optim,

    # For base models, we use the 'text' field and standard language modeling
    dataset_text_field="text",
    max_length=MAX_LENGTH,
    use_liger_kernel=target_liger_kernel,
    # fp16=True,
    bf16=False,
    fp16=DEVICE == "mps",

    activation_offloading=DEVICE == "cuda",
    gradient_checkpointing=True,
    # Disable assistant_only_loss for base models (not using chat format)
)

🍎 CUDA Not Available (likely Mac/MPS): Using MPS-friendly settings (standard optimizer, No Liger).


In [13]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer
);

{'label': 'spam', 'text': 'Instruction: Classify the message as either `valid` or `spam` do not add anything else into the response\n\nEmail: Subject: how have you been , , obtain all of your meds here . . topic snuggle\r\nsave over 50 % onbprescriptionwdrugs\r\nwith our on - linelpharmacy you can\r\n1 . order name right from home ( not the cheap\r\neuropean versions on sites offer )\r\n2 . get it shipped same day to your door step\r\n3 . never have to worry about getting a doctorsto write the prescriptionragain\r\n4 . save hundreds of dollars over your localhpharmacy\r\nif all this sounds good to you then you need to\r\nclick here\r\nfor more about what we offer . we carry everything from vicodin , valum ,\r\nxanax , and viagra . so go to our site to see how much we can save you\r\ntoday .\r\n\n\nClassification: spam'}


Adding EOS to train dataset:   0%|          | 0/2382 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2382 [00:00<?, ? examples/s]

{'label': 'spam', 'text': 'Instruction: Classify the message as either `valid` or `spam` do not add anything else into the response\n\nEmail: Subject: how have you been , , obtain all of your meds here . . topic snuggle\r\nsave over 50 % onbprescriptionwdrugs\r\nwith our on - linelpharmacy you can\r\n1 . order name right from home ( not the cheap\r\neuropean versions on sites offer )\r\n2 . get it shipped same day to your door step\r\n3 . never have to worry about getting a doctorsto write the prescriptionragain\r\n4 . save hundreds of dollars over your localhpharmacy\r\nif all this sounds good to you then you need to\r\nclick here\r\nfor more about what we offer . we carry everything from vicodin , valum ,\r\nxanax , and viagra . so go to our site to see how much we can save you\r\ntoday .\r\n\n\nClassification: spam<|end_of_text|>'}
{'label': 'valid', 'text': 'Instruction: Classify the message as either `valid` or `spam` do not add anything else into the response\n\nEmail: Subject: c

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [14]:
trainer_stats = trainer.train()

/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,3.546700
20,2.557200
30,2.640800
40,2.265400
50,2.632000
60,2.409500
70,2.424000
80,2.522300
90,2.713700
100,2.444500


/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [15]:

def run_mail_classification(email_text: str, max_tokens: int = 10):
    instruction = 'Classify the message as either `valid` or `spam` do not add anything else into the response'
    
    # Format the prompt the same way as training data (without the answer)
    prompt = f"""Instruction: {instruction}

Email: {email_text}

Classification:"""

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Generate response with proper stopping criteria
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,  # Critical: tell model when to stop
        do_sample=False,  # Use greedy decoding for deterministic output
        temperature=None,  # Not needed with do_sample=False
    )

    # Decode only the new tokens (the classification)
    response = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
    return response.strip()


In [16]:
def run_mail_from_dataset_classification(dataset, index):
    # Extract the text from dataset (which already has the prompt format)
    # We need to remove the answer part to create the inference prompt
    full_text = dataset[index]['text']
    
    # Split on 'Classification:' and take everything before it, then add 'Classification:' back
    prompt = full_text.split('Classification:')[0] + 'Classification:'
    
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Generate response with proper stopping criteria
    outputs = model.generate(
        **inputs,
        max_new_tokens=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,  # Critical: tell model when to stop
        do_sample=False,  # Use greedy decoding for deterministic output
        temperature=None,  # Not needed with do_sample=False
    )

    # Decode only the new tokens
    response = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
    return response.strip()


In [17]:
test_dataset = dataset['test']
incorrect_samples = []
correct_samples_count = 0
incorrect_samples_count = 0
for i in range (0, min(len(test_dataset), 500)):
    print(f"--- Run {i} {test_dataset[i]} ---")
    output = run_mail_from_dataset_classification(test_dataset,i)
    output = 'spam' if (output.split()[0].startswith('spam')) else 'valid'
    actual = test_dataset[i]['label']
    is_correct = output == actual
    print(f"Actual: {actual} Output: {output}")
    if is_correct:
        correct_samples_count += 1
    else:
        incorrect_samples_count += 1

    total_samples_count = correct_samples_count + incorrect_samples_count
    print(f"Samples total: {total_samples_count} samples correct: {correct_samples_count} samples incorrect: {incorrect_samples_count} accuracy: {correct_samples_count / total_samples_count}")
    if not is_correct:
        incorrect_samples.append({
            'content': test_dataset[i]['text'],
            'actual': actual,
            'output': output
        })

    print(output)


The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- Run 0 {'label': 'valid', 'text': 'Instruction: Classify the message as either `valid` or `spam` do not add anything else into the response\n\nEmail: Subject: important - to all domestic employees who participate in the enron\r\ncorp savings plan\r\nif you are a participant in the enron corp . savings plan , please read this very important message .\r\nwe understand that you are concerned about the timing of the move to a new savings plan administrator and the restricted access to your investment funds during the upcoming transition period scheduled to take place beginning at 3 : 00 pm cst on october 26 and ending at 8 : 00 am cst on november 20 .\r\nwe have been working with hewitt and northern trust since july . we understand your concerns and are committed to making this transition period as short as possible without jeopardizing the reconciliation of both the plan in total or your account in particular .\r\nremember that the enron corp . savings plan is an investment vehicle for

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Actual: valid Output: valid
Samples total: 1 samples correct: 1 samples incorrect: 0 accuracy: 1.0
valid
--- Run 1 {'label': 'valid', 'text': "Instruction: Classify the message as either `valid` or `spam` do not add anything else into the response\n\nEmail: Subject: re : enerfin meter 980439 for 10 / 00\r\njackie , talk to darren about this . the deal you reference is an hpl deal with\r\ndynegy and i don ' t have access to it . i ' m on the east desk . yesterday i\r\nextended the deal 421415 for the 6 th and 19 th and meredith inserted a path in\r\nunify - tetco to cover the small overflow volume between hpl and ena . the\r\nena / hpl piece is done . the piece between hpl and dynegy is what you need\r\ninserted . thanks\r\njackie young\r\n12 / 15 / 2000 11 : 32 am\r\nto : sherlyn schumack / hou / ect @ ect\r\ncc : victor lamadrid / hou / ect @ ect\r\nsubject : re : enerfin meter 980439 for 10 / 00\r\nsherlyn , i ' ve placed the correct volumes for days 6 and 9 for ena . i ' ll have\r\n

In [18]:
actual_valid_output_spam_count = 0
actual_spam_output_valid_count = 0
unallowed_value_outputs = 0

for sample in incorrect_samples:
    output = sample['output']
    actual = sample['actual']
    if output != 'valid' and output != 'spam':
        unallowed_value_outputs += 1
    else:
        if actual == 'valid':
            actual_valid_output_spam_count += 1
        else:
            actual_spam_output_valid_count += 1
    print("\n-- Sample fail: --")
    print(f"Guess: {output} Actual: {actual}")
    print(sample['content'][0:100])
print(f"actual_valid_output_spam_count: {actual_valid_output_spam_count} actual_spam_output_valid_count: {actual_spam_output_valid_count} unallowed_value_outputs:{unallowed_value_outputs}")


-- Sample fail: --
Guess: valid Actual: spam
Instruction: Classify the message as either `valid` or `spam` do not add anything else into the resp

-- Sample fail: --
Guess: valid Actual: spam
Instruction: Classify the message as either `valid` or `spam` do not add anything else into the resp

-- Sample fail: --
Guess: valid Actual: spam
Instruction: Classify the message as either `valid` or `spam` do not add anything else into the resp

-- Sample fail: --
Guess: valid Actual: spam
Instruction: Classify the message as either `valid` or `spam` do not add anything else into the resp

-- Sample fail: --
Guess: valid Actual: spam
Instruction: Classify the message as either `valid` or `spam` do not add anything else into the resp

-- Sample fail: --
Guess: valid Actual: spam
Instruction: Classify the message as either `valid` or `spam` do not add anything else into the resp

-- Sample fail: --
Guess: valid Actual: spam
Instruction: Classify the message as either `valid` or `spam` do not add

In [19]:
print(run_mail_classification("""Subject: E-mail details of the client.
    Hi Greg, I have received the following contact info from the apache guys: "dan@apache.com", I just wanted to check if this information is correct.
    Best regards, John
""", 20))

validInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstruction


In [20]:
print(run_mail_classification("""Subject: Obsługa języka polskiego.
    Cześć, wydaje mi się, ze język polski nie zostanie poprawnie sklasyfikowany.
    Pozdrawiam, Wojciech
"""))

validInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstruction


In [21]:
print(run_mail_classification("""Subject: Obsługa języka polskiego.
    Kup nanjowszy ajfon za prawie darmo niskie ceny loteria, jesteś tysięcznym uzytkownikiem!
    Pozdrawiam, Wojciech
"""))

spamInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstruction


In [22]:
print(run_mail_classification("""Subject: Free iPhone.
    Hi Greg, You have won a free iPhone. Press the following link to receive your reward: "http://free-iphone.com"
"""))

spamInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstructionInstruction
